# 06 - SageMaker Model Store

Registers and documents the Yelp sentiment model in SageMaker.

This notebook adds the production-facing SageMaker pieces:
- benchmark model
- SageMaker training job
- batch inference
- Model Registry
- Model Card

**Prerequisite:** Run `04_split.ipynb` first. `05_modeling.ipynb` contains the local modeling comparison used to choose Random Forest.

## 0. Install Dependencies

## 1. Setup

Load the shared config, detect the IAM execution role, and create the boto3 + SageMaker session clients.

In [1]:
import importlib
import subprocess
import sys


def is_importable(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except ImportError:
        return False


# Check what's actually missing. Use direct import (not find_spec) so a
# partially-installed parent package surfaces as ImportError, not as a
# ModuleNotFoundError that aborts the cell.
needs_install = []
if not is_importable("sagemaker.sklearn.estimator"):
    needs_install.append("sagemaker>=2,<3")

if needs_install:
    print(f"Installing: {needs_install}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *needs_install],
        check=True,
    )
    print("\nNew packages installed. Restarting kernel so they load cleanly...")
    print("After the kernel restarts, run all cells from the top again.")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception as restart_error:
        print(f"Auto-restart failed ({restart_error}). Restart the kernel manually.")
else:
    print("All dependencies already present. No restart needed.")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


All dependencies already present. No restart needed.


In [2]:
import json
import os
import tarfile
from pathlib import Path
from time import gmtime, strftime
from urllib.parse import urlparse

import boto3
import pandas as pd
from botocore.exceptions import ClientError
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

try:
    import sagemaker
    from sagemaker.sklearn.estimator import SKLearn
except ImportError as error:
    raise ImportError(
        "The SageMaker SDK was installed or upgraded, but this kernel has not loaded it correctly yet. "
        "Restart the kernel, then run the notebook again from the top."
    ) from error

pd.set_option("display.max_columns", None)

In [3]:
default_config = {
    "REGION": "us-east-2",
    "SOURCE_BUCKET": "aai-540-group1-yelp-reviews",
    "FEATURE_COLS": [
        "review_length",
        "word_count",
        "useful",
        "funny",
        "cool",
        "vader_score",
    ],
    "TARGET_COL": "sentiment",
    "RANDOM_STATE": 42,
}

config_path = Path("project_config.json")
if config_path.exists():
    with config_path.open() as f:
        cfg = json.load(f)
else:
    cfg = default_config

REGION = cfg.get("REGION", default_config["REGION"])
SOURCE_BUCKET = cfg.get("SOURCE_BUCKET", default_config["SOURCE_BUCKET"])
FEATURE_COLS = cfg.get("FEATURE_COLS", default_config["FEATURE_COLS"])
TARGET_COL = cfg.get("TARGET_COL", default_config["TARGET_COL"])
RANDOM_STATE = cfg.get("RANDOM_STATE", default_config["RANDOM_STATE"])

# Identifying tags applied to every SageMaker resource created by this notebook —
# useful for cost tracking and filtering in the AWS console.
TAGS = [
    {"Key": "Project", "Value": "yelp-sentiment"},
    {"Key": "Module", "Value": "AAI-540"},
]

# Surfaced in the Model Card. Change if a teammate runs this notebook.
MODEL_CREATOR = "AAI-540 Group 1"

session = boto3.Session(region_name=REGION)
s3 = session.client("s3")
sagemaker_client = session.client("sagemaker")

sm_session = sagemaker.Session(boto_session=session)


def get_role():
    # Option 1: standard SageMaker
    try:
        return sagemaker.get_execution_role()
    except Exception:
        pass

    # Option 2: SageMaker Studio metadata (regular AWS)
    try:
        with open("/opt/ml/metadata/resource-metadata.json") as f:
            meta = json.load(f)
        return meta["ExecutionRoleArn"]
    except Exception:
        pass

    # Option 3: classic notebook instance
    try:
        with open("/opt/ml/metadata/resource-name") as f:
            nb_name = f.read().strip()
        return session.client("sagemaker").describe_notebook_instance(
            NotebookInstanceName=nb_name
        )["RoleArn"]
    except Exception:
        pass

    # Option 4: AWS Academy
    try:
        return session.client("iam").get_role(RoleName="LabRole")["Role"]["Arn"]
    except Exception:
        pass

    raise RuntimeError(
        "Could not detect IAM role automatically. "
        "Paste your role ARN into MANUAL_ROLE_ARN below."
    )


MANUAL_ROLE_ARN = ""  # paste ARN here if auto-detection fails
role = MANUAL_ROLE_ARN if MANUAL_ROLE_ARN else get_role()

project_prefix = "sentiment-model-store"
run_id = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

print("Region:", REGION)
print("Bucket:", SOURCE_BUCKET)
print("Role:", role)
print("Features:", FEATURE_COLS)
print("Target:", TARGET_COL)
print("Run ID:", run_id)

Region: us-east-1
Bucket: aai540-group1-yelp-data
Role: arn:aws:iam::476629097825:role/LabRole
Features: ['review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score']
Target: sentiment
Run ID: 2026-05-29-02-14-22


## 2. Load Splits from S3

Pull the train, validation, test, and production splits written by notebook 04.

In [4]:
def load_split(split_name):
    local_path = f"/tmp/{split_name}.parquet"
    s3.download_file(SOURCE_BUCKET, f"splits/{split_name}.parquet", local_path)
    return pd.read_parquet(local_path)


train_data = load_split("train")
val_data = load_split("validation")
test_data = load_split("test")
prod_data = load_split("production")

print("Split sizes:")
for name, df in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data),
    ("Production", prod_data),
]:
    print(f"{name:<12}: {len(df):,} rows, positive rate {df[TARGET_COL].mean() * 100:.1f}%")

Split sizes:
Train       : 61,201 rows, positive rate 73.5%
Validation  : 15,278 rows, positive rate 73.5%
Test        : 15,278 rows, positive rate 73.5%
Production  : 93,058 rows, positive rate 71.7%


## 3. Benchmark Model

Simple baseline: predict positive when `vader_score > 0`.

In [5]:
def classification_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }


def benchmark_predict(df):
    y_prob = ((df["vader_score"] + 1) / 2).clip(0, 1)
    y_pred = (df["vader_score"] > 0).astype(int)
    return y_pred, y_prob


benchmark_val_pred, benchmark_val_prob = benchmark_predict(val_data)
benchmark_test_pred, benchmark_test_prob = benchmark_predict(test_data)
benchmark_prod_pred, benchmark_prod_prob = benchmark_predict(prod_data)

benchmark_metrics = {
    "validation": classification_metrics(val_data[TARGET_COL], benchmark_val_pred, benchmark_val_prob),
    "test": classification_metrics(test_data[TARGET_COL], benchmark_test_pred, benchmark_test_prob),
    "production": classification_metrics(prod_data[TARGET_COL], benchmark_prod_pred, benchmark_prod_prob),
}

benchmark_results = pd.DataFrame(benchmark_metrics).T
benchmark_results.round(4)

,accuracy,precision,recall,f1,roc_auc
validation,0.8750,0.8673,0.9799,0.9202,0.9084
test,0.8734,0.8665,0.9786,0.9191,0.9070
production,0.8721,0.8620,0.9782,0.9164,0.9147


## 4. Prepare SageMaker Training Inputs

Save CSV versions of the existing splits for SageMaker training.

In [6]:
training_input_prefix = f"{project_prefix}/training-input/{run_id}"


def upload_training_csv(df, split_name):
    local_path = f"/tmp/{split_name}.csv"
    s3_key = f"{training_input_prefix}/{split_name}.csv"
    df[FEATURE_COLS + [TARGET_COL]].to_csv(local_path, index=False)
    s3.upload_file(local_path, SOURCE_BUCKET, s3_key)
    return f"s3://{SOURCE_BUCKET}/{s3_key}"


train_csv_s3 = upload_training_csv(train_data, "train")
validation_csv_s3 = upload_training_csv(val_data, "validation")
test_csv_s3 = upload_training_csv(test_data, "test")

print("Train CSV:", train_csv_s3)
print("Validation CSV:", validation_csv_s3)
print("Test CSV:", test_csv_s3)

Train CSV: s3://aai540-group1-yelp-data/sentiment-model-store/training-input/2026-05-29-02-14-22/train.csv
Validation CSV: s3://aai540-group1-yelp-data/sentiment-model-store/training-input/2026-05-29-02-14-22/validation.csv
Test CSV: s3://aai540-group1-yelp-data/sentiment-model-store/training-input/2026-05-29-02-14-22/test.csv


## 5. Training Script

Random Forest training and batch inference code used by SageMaker.

In [18]:
os.makedirs("scripts", exist_ok=True)

training_script = r'''
import argparse
import json
import os
import time as _time

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def get_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
    }


def load_channel_csv(channel_dir):
    csv_files = [name for name in os.listdir(channel_dir) if name.endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {channel_dir}")
    return pd.read_csv(os.path.join(channel_dir, csv_files[0]))


def train(args):
    feature_cols = args.feature_cols.split(",")
    target_col = args.target_col

    train_df = load_channel_csv(args.train)
    validation_df = load_channel_csv(args.validation)
    test_df = load_channel_csv(args.test)

    # Optional subsampling so scheduled pipeline runs see different training data
    # each fire without changing the input S3 paths.
    #   sample_seed = 0   (default)  → use full training set, deterministic
    #   sample_seed = -1             → derive a fresh seed from epoch time
    #   sample_seed > 0              → reproducible sample with that seed
    if args.sample_seed != 0:
        actual_seed = (
            int(_time.time() * 1000) % (2**31) if args.sample_seed == -1 else args.sample_seed
        )
        before = len(train_df)
        train_df = train_df.sample(frac=args.sample_frac, random_state=actual_seed)
        print(f"Subsampled train_df: {before} -> {len(train_df)} (frac={args.sample_frac}, seed={actual_seed})")

    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth if args.max_depth > 0 else None,
        class_weight="balanced",
        random_state=args.random_state,
        n_jobs=-1,
    )
    model.fit(train_df[feature_cols], train_df[target_col])

    metrics = {}
    for split_name, split_df in {
        "validation": validation_df,
        "test": test_df,
    }.items():
        y_pred = model.predict(split_df[feature_cols])
        y_prob = model.predict_proba(split_df[feature_cols])[:, 1]
        metrics[split_name] = get_metrics(split_df[target_col], y_pred, y_prob)

    print("Evaluation metrics:")
    print(json.dumps(metrics, indent=2))

    # Flat per-line metrics so SageMaker metric_definitions regex can capture
    # them in the training-job console without parsing output.tar.gz.
    for split_name, split_metrics in metrics.items():
        for metric_name, value in split_metrics.items():
            print(f"{split_name}_{metric_name}={value:.6f}")

    model_bundle = {
        "model": model,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "metrics": metrics,
    }
    os.makedirs(args.model_dir, exist_ok=True)
    joblib.dump(model_bundle, os.path.join(args.model_dir, "model.joblib"))

    output_dir = "/opt/ml/output/data"
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "evaluation.json"), "w") as f:
        json.dump(metrics, f, indent=2)


def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))


def input_fn(input_data, content_type):
    if content_type != "text/csv":
        raise ValueError(f"Unsupported content type: {content_type}")
    from io import StringIO
    return pd.read_csv(StringIO(input_data), header=None)


def predict_fn(input_df, model_bundle):
    model = model_bundle["model"]
    feature_cols = model_bundle["feature_cols"]
    input_df.columns = feature_cols[: input_df.shape[1]]
    probabilities = model.predict_proba(input_df[feature_cols])[:, 1]
    predictions = model.predict(input_df[feature_cols])
    return pd.DataFrame({"prediction": predictions, "probability": probabilities})


def output_fn(prediction, accept):
    if accept != "text/csv":
        raise ValueError(f"Unsupported accept type: {accept}")
    return prediction.to_csv(index=False, header=False)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--feature-cols", type=str, required=True)
    parser.add_argument("--target-col", type=str, required=True)
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=0)
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--sample-seed", type=int, default=0)
    parser.add_argument("--sample-frac", type=float, default=1.0)
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--validation", type=str, default=os.environ.get("SM_CHANNEL_VALIDATION"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    train(parser.parse_args())
'''

script_path = Path("scripts/train_sentiment.py")
script_path.write_text(training_script)
print(f"Wrote {script_path}")

Wrote scripts/train_sentiment.py


## 6. Train Model in SageMaker

Launch a managed SageMaker training job with the Random Forest script, exposing per-split metrics to the training-job console via metric definitions.

In [8]:
training_job_name = f"yelp-sentiment-rf-{run_id}"
output_path = f"s3://{SOURCE_BUCKET}/{project_prefix}/training-output"

# Regexes that match the flat per-line prints emitted by train_sentiment.py,
# so these metrics surface natively in the SageMaker training-job console.
metric_definitions = [
    {"Name": "validation:accuracy", "Regex": r"validation_accuracy=([0-9.]+)"},
    {"Name": "validation:precision", "Regex": r"validation_precision=([0-9.]+)"},
    {"Name": "validation:recall", "Regex": r"validation_recall=([0-9.]+)"},
    {"Name": "validation:f1", "Regex": r"validation_f1=([0-9.]+)"},
    {"Name": "validation:roc_auc", "Regex": r"validation_roc_auc=([0-9.]+)"},
    {"Name": "test:accuracy", "Regex": r"test_accuracy=([0-9.]+)"},
    {"Name": "test:precision", "Regex": r"test_precision=([0-9.]+)"},
    {"Name": "test:recall", "Regex": r"test_recall=([0-9.]+)"},
    {"Name": "test:f1", "Regex": r"test_f1=([0-9.]+)"},
    {"Name": "test:roc_auc", "Regex": r"test_roc_auc=([0-9.]+)"},
]

rf_estimator = SKLearn(
    entry_point=str(script_path),
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
    output_path=output_path,
    sagemaker_session=sm_session,
    hyperparameters={
        "feature-cols": ",".join(FEATURE_COLS),
        "target-col": TARGET_COL,
        "n-estimators": 100,
        "max-depth": 0,
        "random-state": RANDOM_STATE,
    },
    metric_definitions=metric_definitions,
    tags=TAGS,
)

rf_estimator.fit(
    {
        "train": train_csv_s3,
        "validation": validation_csv_s3,
        "test": test_csv_s3,
    },
    job_name=training_job_name,
)

model_artifact_s3 = rf_estimator.model_data
training_image = rf_estimator.training_image_uri()

print("Training job:", training_job_name)
print("Model artifact:", model_artifact_s3)
print("Training image:", training_image)

INFO:sagemaker:Creating training-job with name: yelp-sentiment-rf-2026-05-29-02-14-22


2026-05-29 02:14:26 Starting - Starting the training job.

.

.


2026-05-29 02:14:41 Starting - Preparing the instances for training.

...

.


2026-05-29 02:16:55 Training - Training image download completed. Training in progress..

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-05-29 02:16:56,019 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-05-29 02:16:56,024 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-05-29 02:16:56,026 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-29 02:16:56,044 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-05-29 02:16:56,346 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-05-29 02:16:56,349 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-


2026-05-29 02:17:26 Uploading - Uploading generated training model
2026-05-29 02:17:26 Completed - Training job completed


Training seconds: 143
Billable seconds: 143
Training job: yelp-sentiment-rf-2026-05-29-02-14-22
Model artifact: s3://aai540-group1-yelp-data/sentiment-model-store/training-output/yelp-sentiment-rf-2026-05-29-02-14-22/output/model.tar.gz
Training image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


## 7. Compare Benchmark and SageMaker Model

Read the trained model's evaluation metrics from its output artifact and compare them against the VADER-score benchmark.

In [9]:
training_description = sagemaker_client.describe_training_job(
    TrainingJobName=training_job_name
)
training_job_arn = training_description["TrainingJobArn"]


def download_s3_uri(s3_uri, local_path):
    parsed = urlparse(s3_uri)
    s3.download_file(parsed.netloc, parsed.path.lstrip("/"), local_path)


# Read metrics from the training job's output artifact (evaluation.json written
# by the training script to /opt/ml/output/data) rather than unpickling the
# model locally. The container uses sklearn 1.2 while this kernel is on a
# newer version, so joblib.load() on the trained model would raise an
# incompatible-dtype error on the tree node array.
model_artifact_parsed = urlparse(model_artifact_s3)
output_artifact_key = model_artifact_parsed.path.lstrip("/").rsplit("/", 1)[0] + "/output.tar.gz"
output_artifact_s3 = f"s3://{model_artifact_parsed.netloc}/{output_artifact_key}"

output_tar_path = "/tmp/sagemaker_output.tar.gz"
output_extract_dir = Path("/tmp/sagemaker_output_artifact")
output_extract_dir.mkdir(exist_ok=True)

download_s3_uri(output_artifact_s3, output_tar_path)
with tarfile.open(output_tar_path, "r:gz") as tar:
    tar.extractall(output_extract_dir, filter="data")

with (output_extract_dir / "evaluation.json").open() as f:
    sagemaker_model_metrics = json.load(f)

comparison = pd.concat(
    {
        "benchmark_validation": pd.Series(benchmark_metrics["validation"]),
        "sagemaker_validation": pd.Series(sagemaker_model_metrics["validation"]),
        "benchmark_test": pd.Series(benchmark_metrics["test"]),
        "sagemaker_test": pd.Series(sagemaker_model_metrics["test"]),
    },
    axis=1,
).T

print("Training job ARN:", training_job_arn)
print("Output artifact:", output_artifact_s3)
comparison.round(4)

Training job ARN: arn:aws:sagemaker:us-east-1:476629097825:training-job/yelp-sentiment-rf-2026-05-29-02-14-22
Output artifact: s3://aai540-group1-yelp-data/sentiment-model-store/training-output/yelp-sentiment-rf-2026-05-29-02-14-22/output/output.tar.gz


,accuracy,precision,recall,f1,roc_auc
benchmark_validation,0.8750,0.8673,0.9799,0.9202,0.9084
sagemaker_validation,0.9103,0.9314,0.9477,0.9395,0.9528
benchmark_test,0.8734,0.8665,0.9786,0.9191,0.9070
sagemaker_test,0.9057,0.9280,0.9451,0.9364,0.9538


## 8. Batch Inference

Run batch transform on a small production sample.

In [10]:
batch_sample = prod_data[FEATURE_COLS].head(100)
batch_local_path = "/tmp/production_batch_input.csv"
batch_key = f"{project_prefix}/batch-input/{run_id}/production_batch_input.csv"
batch_sample.to_csv(batch_local_path, index=False, header=False)
s3.upload_file(batch_local_path, SOURCE_BUCKET, batch_key)

batch_input_s3 = f"s3://{SOURCE_BUCKET}/{batch_key}"
batch_output_s3 = f"s3://{SOURCE_BUCKET}/{project_prefix}/batch-output/{run_id}/"

transformer = rf_estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=batch_output_s3,
    accept="text/csv",
)

transformer.transform(
    data=batch_input_s3,
    content_type="text/csv",
    split_type="Line",
)
transformer.wait()

print("Batch input:", batch_input_s3)
print("Batch output:", batch_output_s3)

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-05-29-02-17-42-474


INFO:sagemaker:Creating transform job with name: sagemaker-scikit-learn-2026-05-29-02-17-43-821


.

.

.

...

.

.

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-05-29 02:22:59,019 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-05-29 02:22:59,022 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-05-29 02:22:59,023 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/ngi

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-05-29 02:22:59,019 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-05-29 02:22:59,022 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-05-29 02:22:59,023 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/ngi

In [11]:
batch_output_key = f"{project_prefix}/batch-output/{run_id}/production_batch_input.csv.out"
batch_output_local = "/tmp/production_batch_predictions.csv"
s3.download_file(SOURCE_BUCKET, batch_output_key, batch_output_local)

batch_predictions = pd.read_csv(
    batch_output_local,
    header=None,
    names=["prediction", "probability"],
)

batch_predictions.head()

,prediction,probability
0,1,0.99
1,0,0.25
2,1,1.00
3,1,1.00
4,1,0.77


## 9. Create Model Package Group

Create (or reuse) the Model Registry group that versions the sentiment model.

In [12]:
model_package_group_name = "yelp-sentiment-model-group"

try:
    response = sagemaker_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Model registry group for Yelp review sentiment classification.",
        Tags=TAGS,
    )
    model_package_group_arn = response["ModelPackageGroupArn"]
    print("Created model package group:", model_package_group_arn)
except ClientError as error:
    if "already exists" not in str(error).lower():
        raise
    description = sagemaker_client.describe_model_package_group(
        ModelPackageGroupName=model_package_group_name
    )
    model_package_group_arn = description["ModelPackageGroupArn"]
    print("Using existing model package group:", model_package_group_arn)

Using existing model package group: arn:aws:sagemaker:us-east-1:476629097825:model-package-group/yelp-sentiment-model-group


## 10. Register Model Package

Register the trained model as a new versioned package, attaching its evaluation metrics and batch-output location as metadata.

In [14]:
customer_metadata = {
    "project": "yelp-review-sentiment",
    "model_type": "RandomForestClassifier",
    "benchmark": "vader_score_threshold",
    "validation_f1": str(sagemaker_model_metrics["validation"]["f1"]),
    "test_f1": str(sagemaker_model_metrics["test"]["f1"]),
    "test_roc_auc": str(sagemaker_model_metrics["test"]["roc_auc"]),
    "batch_output": batch_output_s3,
}

model_package_response = sagemaker_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription=f"Yelp sentiment Random Forest model trained on run {run_id}.",
    InferenceSpecification={
        "Containers": [
            {
                "Image": training_image,
                "ModelDataUrl": model_artifact_s3,
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large"],
        "SupportedTransformInstanceTypes": ["ml.m5.large"],
    },
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties=customer_metadata,
)

model_package_arn = model_package_response["ModelPackageArn"]
print("Registered model package:", model_package_arn)

Registered model package: arn:aws:sagemaker:us-east-1:476629097825:model-package/yelp-sentiment-model-group/2


## 11. Create Model Card

Document the model's intended use, training data, evaluation, and ethical considerations in a SageMaker Model Card.

In [15]:
model_card_name = f"yelp-sentiment-model-card-{run_id}"

model_card_content = {
    "model_overview": {
        "model_name": "Yelp Sentiment Random Forest",
        "model_description": "Binary classifier that predicts whether a Yelp review is positive or negative using engineered review features.",
        "model_creator": MODEL_CREATOR,
        "problem_type": "Binary Classification",
        "algorithm_type": "Random Forest",
        "model_artifact": [model_artifact_s3],
        "inference_environment": {
            "container_image": [training_image],
        },
    },
    "intended_uses": {
        "purpose_of_model": "Classify Yelp reviews as positive or negative for sentiment analytics.",
        "intended_uses": "Batch scoring of engineered Yelp review records.",
        "factors_affecting_model_efficiency": "Performance depends on feature quality, VADER sentiment score quality, and class distribution over time.",
        "risk_rating": "Medium",
        "explanations_for_risk_rating": "Incorrect sentiment classification can affect downstream business insights, but this model is not used for safety-critical decisions.",
    },
    "business_details": {
        "business_problem": "Summarize customer sentiment from Yelp reviews at scale.",
        "business_stakeholders": "Analytics and customer experience teams.",
        "line_of_business": "Customer analytics",
    },
    "training_details": {
        "training_job_details": {
            "training_arn": training_job_arn,
            "training_datasets": [train_csv_s3, validation_csv_s3, test_csv_s3],
            "training_environment": {
                "container_image": [training_image],
            },
        },
        "training_observations": "Random Forest outperformed the VADER-score benchmark and local logistic regression baseline on F1 score.",
    },
    "evaluation_details": [
        {
            "name": "Test set evaluation",
            "evaluation_observation": "Held-out 2019 test data used for final evaluation.",
            "datasets": [test_csv_s3],
            "metric_groups": [
                {
                    "name": "classification_metrics",
                    "metric_data": [
                        {"name": metric, "type": "number", "value": value}
                        for metric, value in sagemaker_model_metrics["test"].items()
                    ],
                }
            ],
        }
    ],
    "additional_information": {
        "ethical_considerations": "The model is trained on Yelp review text-derived features and may reflect bias in review behavior or language patterns.",
        "caveats_and_recommendations": "Monitor class balance, feature distributions, and prediction quality before using the model for business decisions.",
        "custom_details": {
            "model_package_arn": model_package_arn,
            "model_package_group_name": model_package_group_name,
        },
    },
}

# Let any create error propagate so we can diagnose it. Previously this was
# wrapped in try/except that printed and swallowed the exception, which made
# downstream describe_model_card calls fail with ResourceNotFound.
model_card_response = sagemaker_client.create_model_card(
    ModelCardName=model_card_name,
    Content=json.dumps(model_card_content),
    ModelCardStatus="Draft",
    Tags=TAGS,
)
model_card_arn = model_card_response["ModelCardArn"]
print("Created model card:", model_card_arn)

Created model card: arn:aws:sagemaker:us-east-1:476629097825:model-card/yelp-sentiment-model-card-2026-05-29-02-14-22


## 12. Summary

Recap the registered model, its benchmark comparison, and the key Model Store artifacts (training job, model package, model card, batch output).

In [16]:
summary = pd.DataFrame(
    [
        {"artifact": "training_job", "value": training_job_name},
        {"artifact": "training_job_arn", "value": training_job_arn},
        {"artifact": "model_artifact", "value": model_artifact_s3},
        {"artifact": "batch_input", "value": batch_input_s3},
        {"artifact": "batch_output", "value": batch_output_s3},
        {"artifact": "model_package_group", "value": model_package_group_name},
        {"artifact": "model_package_arn", "value": model_package_arn},
        {"artifact": "model_card_name", "value": model_card_name},
    ]
)

print("Benchmark vs SageMaker model:")
display(comparison.round(4))

print("Model Store artifacts:")
display(summary)

Benchmark vs SageMaker model:


,accuracy,precision,recall,f1,roc_auc
benchmark_validation,0.8750,0.8673,0.9799,0.9202,0.9084
sagemaker_validation,0.9103,0.9314,0.9477,0.9395,0.9528
benchmark_test,0.8734,0.8665,0.9786,0.9191,0.9070
sagemaker_test,0.9057,0.9280,0.9451,0.9364,0.9538


Model Store artifacts:


,artifact,value
0,training_job,yelp-sentiment-rf-2026-05-29-02-14-22
1,training_job_arn,arn:aws:sagemaker:us-east-1:476629097825:train...
2,model_artifact,s3://aai540-group1-yelp-data/sentiment-model-s...
3,batch_input,s3://aai540-group1-yelp-data/sentiment-model-s...
4,batch_output,s3://aai540-group1-yelp-data/sentiment-model-s...
5,model_package_group,yelp-sentiment-model-group
6,model_package_arn,arn:aws:sagemaker:us-east-1:476629097825:model...
7,model_card_name,yelp-sentiment-model-card-2026-05-29-02-14-22
